
# Exogenous Forecast Error Diagnosis

이 notebook은 기존 학습 결과를 다시 학습하지 않고 checkpoint로 불러와 inference를 재현한 뒤,
analytics parquet를 이용해 **왜 이런 오차가 나오는지**를 분해해서 보는 용도입니다.

핵심 질문:
- 과대/과소예측이 어떤 lifecycle stage에서 심한가?
- sparse / erratic / smooth 같은 demand regime에 따라 모델 차이가 어떻게 나는가?
- obsolescence, memory length, spike, seasonality, cluster가 오차에 어떤 영향을 주는가?
- 어떤 segment에서 `patchtst_exo / timexer / exotst` 중 누가 상대적으로 나은가?


In [ ]:

from __future__ import annotations

%load_ext autoreload
%autoreload 2

import json
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import Image, display

import importlib.util

REPO_ROOT_OVERRIDE = None


def _looks_like_repo(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src').exists()


def _iter_named_repo_candidates(root: Path, repo_name: str = 'ts_forecaster_lib', max_depth: int = 4):
    if not root.exists() or not root.is_dir():
        return
    try:
        root_resolved = root.resolve()
    except Exception:
        root_resolved = root
    stack = [(root_resolved, 0)]
    while stack:
        current, depth = stack.pop()
        if current.name == repo_name:
            yield current
        if depth >= max_depth:
            continue
        try:
            children = list(current.iterdir())
        except Exception:
            continue
        for child in children:
            if child.is_dir() and not child.name.startswith('.'):
                stack.append((child, depth + 1))


def find_repo_root(start: Path, explicit_repo_root: Path | None = None) -> Path | None:
    env_repo_root = os.environ.get('TS_FORECASTER_REPO_ROOT')
    candidates = []
    if explicit_repo_root is not None:
        candidates.append(Path(explicit_repo_root).expanduser().resolve())
    if env_repo_root:
        candidates.append(Path(env_repo_root).expanduser().resolve())
    candidates.extend([start, *start.parents])
    home = Path.home()
    common_roots = [
        home,
        home / 'workspace',
        home / 'workspaces',
        home / 'projects',
        home / 'PycharmProjects',
        Path('/workspace'),
        Path('/workspaces'),
        Path('/home'),
        Path('/root'),
    ]
    for root in common_roots:
        candidates.extend(_iter_named_repo_candidates(root))
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if _looks_like_repo(candidate):
            return candidate
    return None


def resolve_import_paths() -> tuple[Path | None, Path | None]:
    repo_root = find_repo_root(Path.cwd().resolve(), explicit_repo_root=REPO_ROOT_OVERRIDE)
    src_root = repo_root / 'src' if repo_root is not None else None
    if src_root is not None and str(src_root) not in sys.path:
        sys.path.insert(0, str(src_root))
    spec = importlib.util.find_spec('modeling_module')
    if spec is None or spec.origin is None:
        raise RuntimeError(
            'Could not import modeling_module. Set REPO_ROOT_OVERRIDE / TS_FORECASTER_REPO_ROOT or install the package.'
        )
    module_init = Path(spec.origin).resolve()
    module_root = module_init.parent
    if repo_root is None and module_root.parent.name == 'src':
        repo_root = module_root.parent.parent
        src_root = repo_root / 'src'
    return repo_root, src_root


NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT, SRC_ROOT = resolve_import_paths()

from model_test.exogenous_test.exogenous_ab_utils import (
    attach_actuals,
    make_point_forecast_result_table,
    compute_metric_tables,
    plot_latest_revision_aggregate,
    plot_latest_revision_part_grid,
    select_latest_revision,
    select_plot_parts,
    ensure_dir,
    resolve_single_plan_week,
    select_eval_ids_with_full_actual_coverage,
)
from model_test.exogenous_test.run_exogenous_model_ab import (
    DATE_COL,
    FUTURE_EXO_CONT_COLS,
    FREQ,
    ID_COL,
    MODEL_SPECS,
    PAST_EXO_CAT_COLS,
    PAST_EXO_CONT_COLS,
    TARGET_EXO_SOURCE,
    TARGET_SOURCE,
    Y_COL,
    CALLBACK_LOOKUP_FUTURE_COLS,
    resolve_model_specs,
)
from model_test.model_test_utils import yyyyww_to_monday
from model_test.total_train.dsio_total_running import (
    EXO_SOURCE_FALLBACK,
    configure_torch_runtime,
    load_polars_table,
    prepare_exo_one_table,
    prepare_target_df,
    resolve_exo_source,
    set_global_seed,
)
from modeling_module import (
    DataColumnConfig,
    DataRequest,
    DataWindowConfig,
    ExogenousConfig,
    LoaderConfig,
    load_predictor,
)
from modeling_module.api.data import build_datamodule

print('REPO_ROOT :', REPO_ROOT)
print('SRC_ROOT  :', SRC_ROOT)



## Config

이 notebook은 학습을 다시 하지 않고 **checkpoint를 불러와 inference만 재현**합니다.
따라서 아래 설정은 기존 AB notebook에서 쓴 값과 맞춰주는 것이 중요합니다.


In [ ]:

artifact_root = REPO_ROOT / 'artifacts' / 'exogenous_test' / 'notebook_ab'
ensure_dir(artifact_root)

models_to_run = ['patchtst_exo', 'timexer', 'exotst']
future_exo_source = 'columns'

lookback = 52
horizon = 26
plan_week = 202541
sample_part_count = 1000
plot_part_count = 20
max_parts_per_plan = 10_000

infer_batch_size = 128
num_workers = 0
pin_memory = False
persistent_workers = False
prefetch_factor = 2
seed = 42

auto_save_diagnosis = True

diag_run_name = f'source_{future_exo_source}__lifecycle_sales_master'
diag_root = ensure_dir(artifact_root / diag_run_name)
diag_plot_dir = ensure_dir(diag_root / 'diagnosis_plots')

use_lifecycle_future_features = True
use_lifecycle_past_features = True
lifecycle_start_mode = 'sales_master'
lifecycle_start_col = 'sales_start_yyyyww'
lifecycle_peak_weeks = 104
lifecycle_tail_weeks = 104
lifecycle_oper_part_path = REPO_ROOT / 'raw_data' / 'raw' / 'tb_mst_oper_part.parquet'
lifecycle_sales_parts_path = REPO_ROOT / 'raw_data' / 'raw' / 'tb_dyn_sales_parts.parquet'

analytics_root = REPO_ROOT / 'raw_data' / 'analytics'
analytics_paths = {
    'acf': analytics_root / 'acf_memory_length_estimator.parquet',
    'calendar': analytics_root / 'calendar_effect_extractor.parquet',
    'change_point': analytics_root / 'change_point_detector.parquet',
    'part_cluster': analytics_root / 'clustering_profiler_part_cluster.parquet',
    'selector': analytics_root / 'forecast_model_selector.parquet',
    'intermittent': analytics_root / 'intermittent_detector.parquet',
    'level': analytics_root / 'level_scale_volatility_profiler.parquet',
    'lifecycle': analytics_root / 'lifecycle_stage_detector.parquet',
    'obsolescence': analytics_root / 'obsolescence_risk_scorer.parquet',
    'outlier': analytics_root / 'outlier_spike_detector.parquet',
    'pattern': analytics_root / 'pattern_similarity_embedding.parquet',
    'seasonality': analytics_root / 'seasonality_detector_weekly.parquet',
    'trend': analytics_root / 'trend_strength_analyzer.parquet',
}

set_global_seed(seed)
default_device, device_note = configure_torch_runtime()
device = default_device

print('diag_root          :', diag_root)
print('models_to_run      :', models_to_run)
print('lookback / horizon :', lookback, horizon)
print('plan_week          :', plan_week)
print('sample_part_count  :', sample_part_count)
print('device             :', device)
if device_note:
    print('device_note        :', device_note)



## Data Preparation

여기서는 기존 AB notebook과 같은 방식으로 target / exo / lifecycle feature를 재구성하고,
동시에 analytics parquet를 part-level profile table로 합칩니다.


In [ ]:


def yyyyww_to_week_index(yyyyww: int) -> int:
    return yyyyww_to_monday(int(yyyyww)).toordinal() // 7


def load_lifecycle_master_frames(selected_ids: list[str]) -> tuple[pl.DataFrame, pl.DataFrame]:
    oper_path = Path(lifecycle_oper_part_path)
    sales_path = Path(lifecycle_sales_parts_path)
    if not oper_path.exists():
        raise FileNotFoundError(f'lifecycle oper-part parquet not found: {oper_path}')
    if not sales_path.exists():
        raise FileNotFoundError(f'lifecycle sales-parts parquet not found: {sales_path}')

    oper_part_df = (
        pl.read_parquet(oper_path)
        .filter(pl.col(ID_COL).cast(pl.String).is_in(selected_ids))
        .select([
            pl.col(ID_COL).cast(pl.String).alias(ID_COL),
            pl.col('demand_start_dt').cast(pl.Int64),
            pl.col('demand_end_dt').cast(pl.Int64),
            pl.col('warranty').cast(pl.Int64),
        ])
        .unique(subset=[ID_COL])
        .with_columns(
            (pl.col('warranty') * 52 / 12).round(0).cast(pl.Int64).alias('warranty_weeks')
        )
    )

    sales_part_df = (
        pl.read_parquet(sales_path)
        .filter(pl.col(ID_COL).cast(pl.String).is_in(selected_ids))
        .filter(pl.col('sales_qty') > 0)
        .filter(pl.col('sales_dt').cast(pl.Int64) < int(plan_week))
        .group_by(pl.col(ID_COL).cast(pl.String).alias(ID_COL))
        .agg(pl.col('sales_dt').cast(pl.Int64).min().alias('first_sales_week'))
        .sort(ID_COL)
    )
    return oper_part_df, sales_part_df


def build_lifecycle_anchor_df(
    target_df: pl.DataFrame,
    exo_df: pl.DataFrame,
    oper_part_df: pl.DataFrame,
    sales_part_df: pl.DataFrame,
    *,
    start_mode: str,
    start_col: str,
) -> pl.DataFrame:
    first_observed = (
        target_df.group_by(ID_COL)
        .agg(pl.col(DATE_COL).min().cast(pl.Int64).alias('first_observed_week'))
    )

    first_positive = (
        target_df
        .filter((pl.col(Y_COL) > 0) & (pl.col(DATE_COL).cast(pl.Int64) < int(plan_week)))
        .group_by(ID_COL)
        .agg(pl.col(DATE_COL).min().cast(pl.Int64).alias('first_positive_week'))
    )

    start_col_df = None
    resolved_mode = str(start_mode)
    if resolved_mode == 'sales_start_col' and start_col in exo_df.columns:
        start_col_df = (
            exo_df.select([
                pl.col(ID_COL).cast(pl.String).alias(ID_COL),
                pl.col(start_col).cast(pl.Int64).alias('sales_start_week'),
            ])
            .drop_nulls(['sales_start_week'])
            .group_by(ID_COL)
            .agg(pl.col('sales_start_week').min().alias('sales_start_week'))
        )
    elif resolved_mode == 'sales_start_col':
        resolved_mode = 'sales_master'

    anchor_df = (
        first_observed
        .join(first_positive, on=ID_COL, how='left')
        .join(oper_part_df, on=ID_COL, how='left')
        .join(sales_part_df, on=ID_COL, how='left')
    )
    if start_col_df is not None:
        anchor_df = anchor_df.join(start_col_df, on=ID_COL, how='left')

    if resolved_mode == 'first_observed':
        start_expr = pl.col('first_observed_week')
    elif resolved_mode == 'first_positive':
        start_expr = pl.coalesce([pl.col('first_positive_week'), pl.col('first_observed_week')])
    elif resolved_mode == 'sales_master':
        start_expr = pl.coalesce([
            pl.col('demand_start_dt'),
            pl.col('first_sales_week'),
            pl.col('first_observed_week'),
        ])
    elif resolved_mode == 'sales_start_col':
        start_expr = pl.coalesce([
            pl.col('sales_start_week'),
            pl.col('demand_start_dt'),
            pl.col('first_sales_week'),
            pl.col('first_observed_week'),
        ])
    else:
        raise ValueError(f'Unsupported lifecycle_start_mode={start_mode!r}')

    return (
        anchor_df
        .with_columns([
            start_expr.cast(pl.Int64).alias('lifecycle_start_week'),
            pl.col('demand_end_dt').cast(pl.Int64).alias('lifecycle_end_week'),
        ])
        .select([
            ID_COL,
            'first_observed_week',
            'first_positive_week',
            'first_sales_week',
            'demand_start_dt',
            'demand_end_dt',
            'warranty_weeks',
            'lifecycle_start_week',
            'lifecycle_end_week',
        ])
    )


def build_lifecycle_feature_frame(
    target_df: pl.DataFrame,
    exo_df: pl.DataFrame,
    oper_part_df: pl.DataFrame,
    sales_part_df: pl.DataFrame,
    *,
    start_mode: str,
    start_col: str,
    peak_weeks: int,
    tail_weeks: int,
) -> pl.DataFrame:
    anchor_df = build_lifecycle_anchor_df(
        target_df,
        exo_df,
        oper_part_df,
        sales_part_df,
        start_mode=start_mode,
        start_col=start_col,
    )

    peak = float(max(int(peak_weeks), 1))
    tail = float(max(int(tail_weeks), 1))

    lifecycle = (
        target_df.select([
            pl.col(ID_COL).cast(pl.String).alias(ID_COL),
            pl.col(DATE_COL).cast(pl.Int64).alias(DATE_COL),
        ])
        .unique()
        .join(anchor_df, on=ID_COL, how='left')
        .with_columns([
            pl.col(DATE_COL).map_elements(lambda x: int(yyyyww_to_week_index(int(x))), return_dtype=pl.Int64).alias('week_idx'),
            pl.when(pl.col('lifecycle_start_week').is_not_null())
            .then(pl.col('lifecycle_start_week').map_elements(lambda x: int(yyyyww_to_week_index(int(x))), return_dtype=pl.Int64))
            .otherwise(None)
            .alias('lifecycle_start_idx'),
            pl.when(pl.col('lifecycle_end_week').is_not_null())
            .then(pl.col('lifecycle_end_week').map_elements(lambda x: int(yyyyww_to_week_index(int(x))), return_dtype=pl.Int64))
            .otherwise(None)
            .alias('lifecycle_end_idx'),
        ])
        .with_columns([
            (pl.col('week_idx') - pl.col('lifecycle_start_idx')).alias('lc_elapsed_raw'),
            (pl.col('lifecycle_end_idx') - pl.col('week_idx')).alias('lc_weeks_to_end_raw'),
            (pl.col('lifecycle_end_idx') - pl.col('lifecycle_start_idx')).alias('lc_total_cycle_weeks'),
        ])
        .with_columns([
            pl.col('lc_elapsed_raw').clip(lower_bound=0).alias('lc_age_week'),
            (pl.col('lc_elapsed_raw') < 0).cast(pl.Float64).alias('lc_before_start_flag'),
            (pl.col('lc_weeks_to_end_raw') < 0).cast(pl.Float64).alias('lc_after_demand_end_flag'),
            ((pl.col('lc_elapsed_raw') >= 0) & (pl.col('lc_weeks_to_end_raw') >= 0)).cast(pl.Float64).alias('lc_in_active_window'),
        ])
        .with_columns([
            (pl.col('lc_age_week') / peak).clip(0.0, 1.0).alias('lc_age_norm_peak'),
            (pl.col('lc_age_week') / (peak + tail)).clip(0.0, 1.0).alias('lc_age_norm_total'),
            (pl.col('lc_age_week') >= peak).cast(pl.Float64).alias('lc_after_peak_flag'),
            ((pl.col('lc_age_week') - peak).clip(lower_bound=0) / tail).clip(0.0, 1.0).alias('lc_decay_norm_tail'),
            pl.col('lc_age_week').map_elements(lambda x: float(math.log1p(max(float(x), 0.0))), return_dtype=pl.Float64).alias('lc_age_log1p'),
        ])
        .with_columns([
            pl.when(pl.col('lc_total_cycle_weeks') > 0)
            .then((pl.col('lc_elapsed_raw') / pl.col('lc_total_cycle_weeks')).clip(0.0, 1.0))
            .otherwise(0.0)
            .alias('lc_demand_progress'),
            pl.col('lc_weeks_to_end_raw').clip(lower_bound=0).alias('lc_weeks_to_end_pos'),
        ])
        .with_columns([
            (pl.col('lc_weeks_to_end_pos') / tail).clip(0.0, 1.0).alias('lc_weeks_to_end_norm'),
            pl.when(pl.col('warranty_weeks') > 0)
            .then((pl.col('lc_age_week') / pl.col('warranty_weeks')).clip(0.0, 1.0))
            .otherwise(0.0)
            .alias('lc_warranty_progress'),
            pl.when(pl.col('warranty_weeks') > 0)
            .then(((pl.col('warranty_weeks') - pl.col('lc_age_week')).clip(lower_bound=0) / pl.col('warranty_weeks')).clip(0.0, 1.0))
            .otherwise(0.0)
            .alias('lc_weeks_to_warranty_end_norm'),
            pl.when(pl.col('warranty_weeks') > 0)
            .then((pl.col('lc_age_week') > pl.col('warranty_weeks')).cast(pl.Float64))
            .otherwise(0.0)
            .alias('lc_after_warranty_flag'),
        ])
        .select([
            ID_COL,
            DATE_COL,
            'lc_age_norm_peak',
            'lc_age_norm_total',
            'lc_after_peak_flag',
            'lc_decay_norm_tail',
            'lc_age_log1p',
            'lc_demand_progress',
            'lc_weeks_to_end_norm',
            'lc_after_demand_end_flag',
            'lc_before_start_flag',
            'lc_in_active_window',
            'lc_warranty_progress',
            'lc_weeks_to_warranty_end_norm',
            'lc_after_warranty_flag',
        ])
        .sort([ID_COL, DATE_COL])
    )
    return lifecycle


def build_lifecycle_augmented_exo(exo_df: pl.DataFrame, lifecycle_df: pl.DataFrame, lifecycle_cols: list[str]) -> pl.DataFrame:
    return (
        exo_df.join(lifecycle_df, on=[ID_COL, DATE_COL], how='left')
        .with_columns([pl.col(c).fill_null(0.0).cast(pl.Float64) for c in lifecycle_cols])
    )


def load_part_profile_df(selected_ids: list[str]) -> pl.DataFrame:
    id_expr = pl.col('oper_part_no').cast(pl.String).alias('oper_part_no')

    intermittent_df = (
        pl.read_parquet(analytics_paths['intermittent'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .select([
            id_expr,
            pl.col('demand_type'),
            pl.col('is_sparsity'),
            pl.col('ADI'),
            pl.col('CV2'),
            pl.col('zero_ratio').alias('zero_ratio_inter'),
            pl.col('nz_mean'),
        ])
    )

    lifecycle_df = (
        pl.read_parquet(analytics_paths['lifecycle'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .select([
            id_expr,
            pl.col('stage'),
            pl.col('stage_code'),
            pl.col('age_periods'),
            pl.col('inactive_gap'),
            pl.col('recent_mean').alias('lifecycle_recent_mean'),
            pl.col('prev_mean').alias('lifecycle_prev_mean'),
        ])
    )

    obsolescence_df = (
        pl.read_parquet(analytics_paths['obsolescence'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .select([
            id_expr,
            pl.col('obsolescence_score'),
            pl.col('time_since_last_demand'),
            pl.col('hazard_like_score'),
            pl.col('recent_zero_ratio'),
        ])
    )

    spike_df = (
        pl.read_parquet(analytics_paths['outlier'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .select([
            id_expr,
            pl.col('outlier_count'),
            pl.col('spike_score'),
        ])
    )

    acf_df = (
        pl.read_parquet(analytics_paths['acf'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .select([
            id_expr,
            pl.col('acf_decay_lag'),
            pl.col('sig_lag_count'),
            pl.col('max_acf'),
        ])
    )

    seasonality_df = (
        pl.read_parquet(analytics_paths['seasonality'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .group_by(pl.col('oper_part_no').cast(pl.String).alias('oper_part_no'))
        .agg([
            pl.col('best_period').drop_nulls().first().alias('best_period'),
            pl.col('best_strength').drop_nulls().first().alias('best_strength'),
        ])
    )

    calendar_df = (
        pl.read_parquet(analytics_paths['calendar'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .select([
            id_expr,
            pl.col('calendar_strength'),
            pl.col('calendar_peak_count'),
            pl.col('calendar_flag'),
        ])
    )

    trend_df = (
        pl.read_parquet(analytics_paths['trend'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .select([
            id_expr,
            pl.col('trend_direction'),
            pl.col('trend_slope'),
            pl.col('trend_strength'),
            pl.col('monotonicity'),
        ])
    )

    level_df = (
        pl.read_parquet(analytics_paths['level'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .select([
            id_expr,
            pl.col('mean').alias('level_mean'),
            pl.col('median').alias('level_median'),
            pl.col('std').alias('level_std'),
            pl.col('p95'),
            pl.col('cv').alias('level_cv'),
            pl.col('volatility_index'),
        ])
    )

    cluster_df = (
        pl.read_parquet(analytics_paths['part_cluster'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .select([
            id_expr,
            pl.col('cluster_id'),
        ])
    )

    selector_df = (
        pl.read_parquet(analytics_paths['selector'])
        .filter(pl.col('oper_part_no').cast(pl.String).is_in(selected_ids))
        .select([
            id_expr,
            pl.col('season_flag'),
            pl.col('calendar_flag').alias('selector_calendar_flag'),
            pl.col('spike_flag'),
            pl.col('regime_flag'),
        ])
    )

    part_profile_df = intermittent_df
    for frame in [lifecycle_df, obsolescence_df, spike_df, acf_df, seasonality_df, calendar_df, trend_df, level_df, cluster_df, selector_df]:
        part_profile_df = part_profile_df.join(frame, on='oper_part_no', how='left')

    return part_profile_df.sort('oper_part_no')


target_raw = load_polars_table(TARGET_SOURCE, 'tb_master_target')

target_df = prepare_target_df(
    target_raw,
    id_col=ID_COL,
    date_col=DATE_COL,
    y_col=Y_COL,
    use_id_sample=False,
    max_ids=sample_part_count,
    sample_part_count=sample_part_count,
    min_obs=lookback + horizon,
    seed=seed,
)

exo_source = resolve_exo_source(TARGET_EXO_SOURCE, EXO_SOURCE_FALLBACK)
exo_raw = load_polars_table(exo_source, exo_source.name)
selected_ids = target_df.select(pl.col(ID_COL).cast(pl.String)).unique().sort(ID_COL).get_column(ID_COL).to_list()
oper_part_df, sales_part_df = load_lifecycle_master_frames(selected_ids)

lifecycle_feature_cols = [
    'lc_age_norm_peak',
    'lc_age_norm_total',
    'lc_after_peak_flag',
    'lc_decay_norm_tail',
    'lc_age_log1p',
    'lc_demand_progress',
    'lc_weeks_to_end_norm',
    'lc_after_demand_end_flag',
    'lc_before_start_flag',
    'lc_in_active_window',
    'lc_warranty_progress',
    'lc_weeks_to_warranty_end_norm',
    'lc_after_warranty_flag',
]

lifecycle_feature_frame = build_lifecycle_feature_frame(
    target_df,
    exo_raw,
    oper_part_df,
    sales_part_df,
    start_mode=lifecycle_start_mode,
    start_col=lifecycle_start_col,
    peak_weeks=lifecycle_peak_weeks,
    tail_weeks=lifecycle_tail_weeks,
)

exo_raw_effective = build_lifecycle_augmented_exo(exo_raw, lifecycle_feature_frame, lifecycle_feature_cols)
future_exo_cont_cols_effective = list(FUTURE_EXO_CONT_COLS) + list(lifecycle_feature_cols)
past_exo_cont_cols_effective = list(PAST_EXO_CONT_COLS) + list(lifecycle_feature_cols)

exo_one_table = prepare_exo_one_table(
    target_df=target_df,
    exo_df=exo_raw_effective,
    id_col=ID_COL,
    date_col=DATE_COL,
    y_col=Y_COL,
    past_exo_cont_cols=list(past_exo_cont_cols_effective),
    future_exo_cont_cols=list(future_exo_cont_cols_effective),
    past_exo_cat_cols=list(PAST_EXO_CAT_COLS),
)

plan_week = resolve_single_plan_week(target_df, date_col=DATE_COL, horizon=horizon, plan_week=plan_week)
train_cutoff = int(plan_week)
train_one_table = exo_one_table.filter(pl.col(DATE_COL) < train_cutoff)
eval_ids = select_eval_ids_with_full_actual_coverage(
    target_df,
    id_col=ID_COL,
    date_col=DATE_COL,
    plan_weeks=[plan_week],
    horizon=horizon,
)

eval_target_df = target_df.filter(pl.col(ID_COL).cast(pl.String).is_in(eval_ids))
infer_one_table = exo_one_table.filter(pl.col(ID_COL).cast(pl.String).is_in(eval_ids))
part_profile_df = load_part_profile_df(eval_ids)

print('target_raw shape :', target_raw.shape)
print('target_df shape  :', target_df.shape)
print('exo_raw shape    :', exo_raw.shape)
print('exo_one_table    :', exo_one_table.shape)
print('train_one_table  :', train_one_table.shape)
print('eval_target_df   :', eval_target_df.shape)
print('infer_one_table  :', infer_one_table.shape)
print('part_profile_df  :', part_profile_df.shape)
print('eval_id_count    :', len(eval_ids))
print('plan_week        :', plan_week)
print('train_cutoff     :', train_cutoff)

display(part_profile_df.head(3))



## Rebuild Forecasts From Saved Checkpoints

이 섹션은 기존 학습 산출물에서 checkpoint를 찾아 같은 inference 조건으로 예측을 재구성합니다.
checkpoint가 없으면 먼저 AB notebook에서 해당 run을 한 번 저장해야 합니다.


In [ ]:


def build_notebook_data_request(
    *,
    df: pl.DataFrame,
    lookback: int,
    horizon: int,
    batch_size: int,
    num_workers: int,
    pin_memory: bool,
    persistent_workers: bool,
    prefetch_factor: int,
    shuffle: bool,
    use_future_exogenous: bool,
):
    future_cols = list(future_exo_cont_cols_effective) if use_future_exogenous else []
    return DataRequest(
        df=df,
        window=DataWindowConfig(lookback=lookback, horizon=horizon, freq=FREQ),
        columns=DataColumnConfig(id_col=ID_COL, date_col=DATE_COL, y_col=Y_COL),
        exogenous=ExogenousConfig(
            use_exogenous_mode=True,
            use_past_exogenous=True,
            use_future_exogenous=use_future_exogenous,
            past_exo_cont_cols=list(past_exo_cont_cols_effective),
            past_exo_cat_cols=list(PAST_EXO_CAT_COLS),
            future_exo_cont_cols=future_cols,
            future_exo_cb=None,
            part_future_exo_fn=None,
        ),
        loader=LoaderConfig(
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
            persistent_workers=persistent_workers,
            prefetch_factor=prefetch_factor,
        ),
    )


def resolve_checkpoint_for_spec(spec) -> Path:
    model_dir = diag_root / spec.label
    manifest_path = model_dir / 'training_manifest.json'
    if manifest_path.exists():
        payload = json.loads(manifest_path.read_text())
        for key in ['primary_ckpt_path', 'best_ckpt_path']:
            value = payload.get(key)
            if value and Path(value).exists():
                return Path(value)
        for key in ['ckpt_paths', 'artifact_ckpt_paths', 'checkpoints']:
            value = payload.get(key)
            if isinstance(value, dict):
                for v in value.values():
                    if v and Path(v).exists():
                        return Path(v)

    patterns = {
        'patchtst_exo': ['*PatchTST*.pt'],
        'timexer': ['*TimeXer*.pt'],
        'exotst': ['*ExoTST*.pt'],
    }
    for pat in patterns.get(spec.label, ['*.pt']):
        matches = sorted(model_dir.glob(pat))
        matches = [m for m in matches if 'Quantile' not in m.name]
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f'No checkpoint found for {spec.label} under {model_dir}. '
        'Run exogenous_model_ab_test.ipynb first or adjust diag_root.'
    )


def infer_one_model(spec):
    ckpt_path = resolve_checkpoint_for_spec(spec)
    predictor = load_predictor(
        str(ckpt_path),
        device=device,
        forecaster_kwargs={
            'target_channel': 0,
            'fill_mode': 'copy_last',
            'use_winsor': True,
            'use_multi_guard': True,
        },
    )
    infer_req = build_notebook_data_request(
        df=infer_one_table,
        lookback=lookback,
        horizon=horizon,
        batch_size=infer_batch_size,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
        shuffle=False,
        use_future_exogenous=spec.use_future_exogenous,
    )
    infer_dm = build_datamodule(infer_req)
    infer_loader = infer_dm.get_inference_loader_at_plan(int(plan_week))
    forecast_df = make_point_forecast_result_table(
        inference_loader=infer_loader,
        predictor=predictor,
        model_name=spec.label,
        plan_week=int(plan_week),
        horizon=horizon,
        device=device,
        max_parts=max_parts_per_plan,
    )
    forecast_df = attach_actuals(
        forecast_df,
        eval_target_df,
        id_col=ID_COL,
        date_col=DATE_COL,
        y_col=Y_COL,
    )
    return ckpt_path, forecast_df


forecast_tables = []
ckpt_map = {}
for spec in resolve_model_specs(models_to_run):
    ckpt_path, forecast_df = infer_one_model(spec)
    ckpt_map[spec.label] = str(ckpt_path)
    forecast_tables.append(forecast_df)
    print(spec.label, 'checkpoint:', ckpt_path)
    print(spec.label, 'forecast shape:', forecast_df.shape)

combined_forecast_df = pl.concat(forecast_tables, how='vertical_relaxed') if forecast_tables else pl.DataFrame()
latest_df = select_latest_revision(combined_forecast_df.filter(pl.col('actual').is_not_null()))

if auto_save_diagnosis and combined_forecast_df.height > 0:
    combined_path = diag_root / 'diagnosis_combined_forecast.parquet'
    latest_path = diag_root / 'diagnosis_latest_forecast.parquet'
    combined_forecast_df.write_parquet(combined_path)
    latest_df.write_parquet(latest_path)
    print('saved:', combined_path)
    print('saved:', latest_path)

combined_forecast_df.head()



## Baseline Metrics and Forecast Plots

먼저 기존 AB notebook과 동일한 수준의 metric / aggregate / part plot을 다시 확인합니다.


In [ ]:


def summarize_scale(df: pl.DataFrame, group_cols: list[str]) -> pl.DataFrame:
    if df.height == 0:
        return pl.DataFrame()
    valid = df.filter(pl.col('actual').is_not_null())
    if valid.height == 0:
        return pl.DataFrame()
    return (
        valid.group_by(group_cols)
        .agg(
            pl.len().alias('n_rows'),
            pl.col('prediction').mean().alias('pred_mean'),
            pl.col('prediction').median().alias('pred_p50'),
            pl.col('prediction').max().alias('pred_max'),
            pl.col('actual').mean().alias('actual_mean'),
            pl.col('actual').median().alias('actual_p50'),
            pl.col('actual').max().alias('actual_max'),
            pl.col('prediction').sum().alias('pred_sum'),
            pl.col('actual').sum().alias('actual_sum'),
        )
        .with_columns(
            pl.when(pl.col('actual_sum').abs() > 1e-12)
            .then(pl.col('pred_sum') / pl.col('actual_sum'))
            .otherwise(None)
            .alias('pred_to_actual_ratio')
        )
        .sort(group_cols)
    )


overall_df, by_horizon_df, latest_summary_df = compute_metric_tables(combined_forecast_df)
scale_df = summarize_scale(latest_df, group_cols=['model_name'])

plot_dir = ensure_dir(diag_root / 'diagnosis_plots')
aggregate_plot_path = plot_dir / 'aggregate_latest_revision.png'
part_grid_plot_path = plot_dir / 'parts_latest_revision.png'
plot_latest_revision_aggregate(latest_df, aggregate_plot_path)
plot_latest_revision_part_grid(
    latest_df,
    sampled_parts=select_plot_parts(latest_df, plot_part_count=plot_part_count),
    save_path=part_grid_plot_path,
    ncols=3,
)

display(latest_summary_df if latest_summary_df.height > 0 else overall_df)
display(scale_df)
if aggregate_plot_path.exists():
    display(Image(filename=str(aggregate_plot_path)))
if part_grid_plot_path.exists():
    display(Image(filename=str(part_grid_plot_path)))



## Error Decomposition By Analytics Segments

여기서부터가 핵심입니다. 예측 오차를 analytics parquet 기준으로 잘라서,
`왜 이런 결과가 생기는지`를 cohort 수준에서 확인합니다.


In [ ]:


def add_quantile_bucket(df: pl.DataFrame, col: str, out_col: str, labels: list[str]) -> pl.DataFrame:
    valid = df.select(pl.col(col).drop_nulls())
    if valid.height == 0:
        return df.with_columns(pl.lit('missing').alias(out_col))
    qs = valid.select([
        pl.col(col).quantile(0.25).alias('q1'),
        pl.col(col).quantile(0.50).alias('q2'),
        pl.col(col).quantile(0.75).alias('q3'),
    ]).to_dicts()[0]
    q1, q2, q3 = qs['q1'], qs['q2'], qs['q3']
    return df.with_columns(
        pl.when(pl.col(col).is_null()).then(pl.lit('missing'))
        .when(pl.col(col) <= q1).then(pl.lit(labels[0]))
        .when(pl.col(col) <= q2).then(pl.lit(labels[1]))
        .when(pl.col(col) <= q3).then(pl.lit(labels[2]))
        .otherwise(pl.lit(labels[3]))
        .alias(out_col)
    )


def summarize_group_error(df: pl.DataFrame, group_col: str) -> pl.DataFrame:
    valid = df.filter(pl.col('actual').is_not_null() & pl.col(group_col).is_not_null())
    if valid.height == 0:
        return pl.DataFrame()
    return (
        valid.group_by(['model_name', group_col])
        .agg([
            pl.len().alias('n_rows'),
            pl.col('part_no').n_unique().alias('n_parts'),
            pl.col('prediction').sum().alias('pred_sum'),
            pl.col('actual').sum().alias('actual_sum'),
            (pl.col('prediction') - pl.col('actual')).mean().alias('bias'),
            (pl.col('prediction') - pl.col('actual')).abs().mean().alias('mae'),
        ])
        .with_columns([
            pl.when(pl.col('actual_sum').abs() > 1e-12)
            .then(pl.col('pred_sum') / pl.col('actual_sum'))
            .otherwise(None)
            .alias('pred_to_actual_ratio'),
            pl.when(pl.col('actual_sum').abs() > 1e-12)
            .then((pl.col('pred_sum') - pl.col('actual_sum')).abs() / pl.col('actual_sum'))
            .otherwise(None)
            .alias('wape_like'),
        ])
        .sort(['model_name', group_col])
    )


def summarize_part_level(df: pl.DataFrame) -> pl.DataFrame:
    valid = df.filter(pl.col('actual').is_not_null())
    return (
        valid.group_by(['model_name', 'part_no'])
        .agg([
            pl.col('prediction').sum().alias('pred_sum'),
            pl.col('actual').sum().alias('actual_sum'),
            (pl.col('prediction') - pl.col('actual')).mean().alias('bias_mean'),
            (pl.col('prediction') - pl.col('actual')).abs().mean().alias('mae'),
        ])
        .with_columns(
            pl.when(pl.col('actual_sum').abs() > 1e-12)
            .then(pl.col('pred_sum') / pl.col('actual_sum'))
            .otherwise(None)
            .alias('pred_to_actual_ratio')
        )
    )


analysis_df = latest_df.join(part_profile_df, left_on='part_no', right_on='oper_part_no', how='left')
analysis_df = add_quantile_bucket(analysis_df, 'obsolescence_score', 'obsolescence_bucket', ['Q1_low', 'Q2', 'Q3', 'Q4_high'])
analysis_df = add_quantile_bucket(analysis_df, 'spike_score', 'spike_bucket', ['Q1_low', 'Q2', 'Q3', 'Q4_high'])
analysis_df = add_quantile_bucket(analysis_df, 'acf_decay_lag', 'memory_bucket', ['short', 'mid_short', 'mid_long', 'long'])
analysis_df = add_quantile_bucket(analysis_df, 'p95', 'scale_bucket', ['small', 'mid_small', 'mid_large', 'large'])

part_level_df = summarize_part_level(analysis_df).join(part_profile_df, left_on='part_no', right_on='oper_part_no', how='left')
part_level_df = add_quantile_bucket(part_level_df, 'obsolescence_score', 'obsolescence_bucket', ['Q1_low', 'Q2', 'Q3', 'Q4_high'])
part_level_df = add_quantile_bucket(part_level_df, 'spike_score', 'spike_bucket', ['Q1_low', 'Q2', 'Q3', 'Q4_high'])
part_level_df = add_quantile_bucket(part_level_df, 'acf_decay_lag', 'memory_bucket', ['short', 'mid_short', 'mid_long', 'long'])
part_level_df = add_quantile_bucket(part_level_df, 'p95', 'scale_bucket', ['small', 'mid_small', 'mid_large', 'large'])

stage_error_df = summarize_group_error(analysis_df, 'stage')
demand_type_error_df = summarize_group_error(analysis_df, 'demand_type')
obsolescence_error_df = summarize_group_error(analysis_df, 'obsolescence_bucket')
spike_error_df = summarize_group_error(analysis_df, 'spike_bucket')
memory_error_df = summarize_group_error(analysis_df, 'memory_bucket')
cluster_error_df = summarize_group_error(analysis_df, 'cluster_id')
scale_error_df = summarize_group_error(analysis_df, 'scale_bucket')

print('stage x model')
display(stage_error_df)
print('demand_type x model')
display(demand_type_error_df)
print('obsolescence bucket x model')
display(obsolescence_error_df)
print('spike bucket x model')
display(spike_error_df)
print('memory bucket x model')
display(memory_error_df)
print('cluster x model')
display(cluster_error_df)
print('scale bucket x model')
display(scale_error_df)



## Which Model Wins Which Segment?

part 단위 horizon 합산 기준으로 어떤 segment에서 어느 모델이 상대적으로 나은지 봅니다.


In [ ]:


best_model_by_part = (
    part_level_df
    .filter(pl.col('actual_sum').abs() > 1e-12)
    .with_columns(((pl.col('pred_sum') - pl.col('actual_sum')).abs() / pl.col('actual_sum')).alias('part_wape'))
    .sort(['part_no', 'part_wape'])
    .group_by('part_no', maintain_order=True)
    .first()
)

print('best model by lifecycle stage')
display(best_model_by_part.group_by(['stage', 'model_name']).len().sort(['stage', 'len'], descending=[False, True]))
print('best model by demand type')
display(best_model_by_part.group_by(['demand_type', 'model_name']).len().sort(['demand_type', 'len'], descending=[False, True]))
print('best model by cluster')
display(best_model_by_part.group_by(['cluster_id', 'model_name']).len().sort(['cluster_id', 'len'], descending=[False, True]))

stage_plot_df = (
    best_model_by_part
    .group_by(['stage', 'model_name'])
    .len()
    .sort(['stage', 'model_name'])
)

stages = stage_plot_df.get_column('stage').unique().to_list() if stage_plot_df.height > 0 else []
models = stage_plot_df.get_column('model_name').unique().to_list() if stage_plot_df.height > 0 else []

if stages and models:
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(stages))
    width = 0.8 / max(1, len(models))
    for idx, model_name in enumerate(models):
        vals = []
        for stage in stages:
            sub = stage_plot_df.filter((pl.col('stage') == stage) & (pl.col('model_name') == model_name))
            vals.append(int(sub.get_column('len')[0]) if sub.height else 0)
        ax.bar(x + idx * width, vals, width=width, label=model_name)
    ax.set_xticks(x + width * (len(models) - 1) / 2)
    ax.set_xticklabels(stages, rotation=20)
    ax.set_title('Best Model Count by Lifecycle Stage')
    ax.set_ylabel('n_parts')
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

if auto_save_diagnosis:
    part_level_path = diag_root / 'diagnosis_part_level.parquet'
    best_model_path = diag_root / 'diagnosis_best_model_by_part.parquet'
    part_level_df.write_parquet(part_level_path)
    best_model_by_part.write_parquet(best_model_path)
    print('saved:', part_level_path)
    print('saved:', best_model_path)



## Segment Calibration

진단 결과 가장 문제가 컸던 `small + erratic + high obsolescence` 세그먼트에 대해
모델별 scale factor를 추정하고, calibration 전후 metric과 aggregate plot을 비교합니다.

주의:
- 이건 학습 자체를 바꾸는 게 아니라 **post-calibration**입니다.
- factor는 현재 backtest segment에서 `actual_sum / pred_sum`으로 추정합니다.
- 필요하면 `require_tail_stage=True`로 두어 `decline/inactive`까지 함께 만족하는 part만 calibration 대상으로 제한할 수 있습니다.


In [ ]:

require_tail_stage = False
factor_clip_min = 0.05
factor_clip_max = 1.00
save_calibrated_outputs = True


def _build_segment_part_df(part_level_df: pl.DataFrame) -> pl.DataFrame:
    out = (
        part_level_df
        .select([
            'part_no',
            'stage',
            'demand_type',
            'obsolescence_bucket',
            'spike_bucket',
            'memory_bucket',
            'scale_bucket',
        ])
        .unique(subset=['part_no'])
    )
    mask = (
        (pl.col('scale_bucket') == 'small') &
        (pl.col('demand_type') == 'erratic') &
        (pl.col('obsolescence_bucket') == 'Q4_high')
    )
    if require_tail_stage:
        mask = mask & pl.col('stage').is_in(['decline', 'inactive'])
    return out.with_columns(mask.alias('target_segment'))


segment_part_df = _build_segment_part_df(part_level_df)
segment_part_summary = (
    segment_part_df
    .group_by(['target_segment', 'stage', 'demand_type', 'scale_bucket', 'obsolescence_bucket'])
    .len()
    .sort('len', descending=True)
)

latest_with_segment = latest_df.join(
    segment_part_df.select(['part_no', 'target_segment']),
    on='part_no',
    how='left',
).with_columns(pl.col('target_segment').fill_null(False))

calibration_factor_df = (
    latest_with_segment
    .filter(pl.col('target_segment'))
    .group_by('model_name')
    .agg([
        pl.col('prediction').sum().alias('segment_pred_sum'),
        pl.col('actual').sum().alias('segment_actual_sum'),
        pl.len().alias('segment_rows'),
        pl.col('part_no').n_unique().alias('segment_parts'),
    ])
    .with_columns(
        pl.when(pl.col('segment_pred_sum').abs() > 1e-12)
        .then((pl.col('segment_actual_sum') / pl.col('segment_pred_sum')).clip(factor_clip_min, factor_clip_max))
        .otherwise(1.0)
        .alias('segment_scale_factor')
    )
    .sort('model_name')
)

calibrated_combined_df = (
    combined_forecast_df
    .join(segment_part_df.select(['part_no', 'target_segment']), on='part_no', how='left')
    .with_columns(pl.col('target_segment').fill_null(False))
    .join(calibration_factor_df.select(['model_name', 'segment_scale_factor']), on='model_name', how='left')
    .with_columns(pl.col('segment_scale_factor').fill_null(1.0))
    .with_columns(
        pl.when(pl.col('target_segment'))
        .then(pl.col('prediction') * pl.col('segment_scale_factor'))
        .otherwise(pl.col('prediction'))
        .alias('prediction')
    )
    .drop(['target_segment', 'segment_scale_factor'])
)

raw_latest_df = select_latest_revision(combined_forecast_df.filter(pl.col('actual').is_not_null()))
cal_latest_df = select_latest_revision(calibrated_combined_df.filter(pl.col('actual').is_not_null()))

raw_metrics_df = (compute_metric_tables(combined_forecast_df)[2]).with_columns(pl.lit('raw').alias('variant'))
cal_metrics_df = (compute_metric_tables(calibrated_combined_df)[2]).with_columns(pl.lit('segment_calibrated').alias('variant'))
metric_compare_df = pl.concat([raw_metrics_df, cal_metrics_df], how='vertical_relaxed').sort(['model_name', 'variant'])

raw_scale_df = summarize_scale(raw_latest_df, group_cols=['model_name']).with_columns(pl.lit('raw').alias('variant'))
cal_scale_df = summarize_scale(cal_latest_df, group_cols=['model_name']).with_columns(pl.lit('segment_calibrated').alias('variant'))
scale_compare_df = pl.concat([raw_scale_df, cal_scale_df], how='vertical_relaxed').sort(['model_name', 'variant'])

raw_seg_df = summarize_group_error(
    raw_latest_df.join(segment_part_df.select(['part_no', 'target_segment']), on='part_no', how='left').with_columns(pl.col('target_segment').fill_null(False)),
    'target_segment',
).with_columns(pl.lit('raw').alias('variant'))
cal_seg_df = summarize_group_error(
    cal_latest_df.join(segment_part_df.select(['part_no', 'target_segment']), on='part_no', how='left').with_columns(pl.col('target_segment').fill_null(False)),
    'target_segment',
).with_columns(pl.lit('segment_calibrated').alias('variant'))
segment_compare_df = pl.concat([raw_seg_df, cal_seg_df], how='vertical_relaxed').sort(['variant', 'model_name', 'target_segment'])

print('segment definition summary')
display(segment_part_summary.head(20))
print('model-specific calibration factors')
display(calibration_factor_df)
print('overall metric compare')
display(metric_compare_df)
print('overall scale compare')
display(scale_compare_df)
print('target segment before/after compare')
display(segment_compare_df)

compare_plot_df = pl.concat([
    raw_latest_df.with_columns(pl.lit('raw').alias('variant')),
    cal_latest_df.with_columns(pl.lit('segment_calibrated').alias('variant')),
], how='vertical_relaxed')

models = sorted(compare_plot_df.get_column('model_name').unique().to_list()) if compare_plot_df.height else []
if models:
    fig, axes = plt.subplots(nrows=len(models), ncols=1, figsize=(14, 4 * len(models)), squeeze=False)
    axes_flat = axes.flatten()
    for ax, model_name in zip(axes_flat, models):
        sub = compare_plot_df.filter(pl.col('model_name') == model_name)
        actual_df = (
            sub.select(['part_no', 'forecast_week', 'actual'])
            .unique(subset=['part_no', 'forecast_week'])
            .group_by('forecast_week')
            .agg(pl.col('actual').sum().alias('actual_sum'))
            .sort('forecast_week')
        )
        x = [str(v) for v in actual_df.get_column('forecast_week').to_list()]
        ax.plot(x, actual_df.get_column('actual_sum').to_list(), marker='o', linewidth=2.2, label='GT', color='#ff6b6b')
        for variant in ['raw', 'segment_calibrated']:
            pred_df = (
                sub.filter(pl.col('variant') == variant)
                .group_by('forecast_week')
                .agg(pl.col('prediction').sum().alias('pred_sum'))
                .sort('forecast_week')
            )
            ax.plot([str(v) for v in pred_df.get_column('forecast_week').to_list()], pred_df.get_column('pred_sum').to_list(), marker='o', label=variant)
        ax.set_title(f'{model_name}: raw vs calibrated aggregate')
        ax.set_xlabel('forecast_week')
        ax.set_ylabel('sum_qty')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

if save_calibrated_outputs:
    calibrated_path = diag_root / 'diagnosis_calibrated_combined_forecast.parquet'
    factors_path = diag_root / 'diagnosis_segment_calibration_factors.parquet'
    calibrated_combined_df.write_parquet(calibrated_path)
    calibration_factor_df.write_parquet(factors_path)
    print('saved:', calibrated_path)
    print('saved:', factors_path)
